In [1]:
from pathlib import Path
from typing import Set

from data_labeling_pipeline.grandtour_missions import missions

grand_tour_folder = Path("/datahome/lmertens/grand_tour_dataset")
output_root = Path("./output")

all_mission_short_names: Set[str] = set([m["short_name"] for m in missions])
mission_short_names_to_process: Set[str] = set(all_mission_short_names)
# Example subset:
# mission_short_names_to_process = {"ETH-1", "SNOW-2"}

missions_to_process = [
    mission
    for mission in missions
    if mission["short_name"] in mission_short_names_to_process
]

print(f"Configured {len(missions_to_process)} missions for processing.")

Configured 49 missions for processing.


In [2]:
from tqdm.notebook import tqdm

from data_labeling_pipeline import DataLabelingPipeline

skip_nonempty_output_dirs = True

processed_mission_outputs = []
erroneous_missions = []

for mission in tqdm(missions_to_process, desc="Processing missions"):
    mission_short_name = mission["short_name"]
    mission_path = grand_tour_folder / "ros1" / mission_short_name
    if not mission_path.exists():
        print(f"Skipping {mission_short_name}: missing {mission_path}")
        continue

    mission_output_dir = output_root / mission_short_name

    if (
        skip_nonempty_output_dirs
        and mission_output_dir.exists()
        and any(mission_output_dir.iterdir())
    ):
        print(f"Skipping {mission_short_name}: Output dir is nonempty")
        continue

    print(f"Processing {mission_short_name} -> {mission_output_dir}")
    pipeline = DataLabelingPipeline(mission_path=mission_path)
    try:
        pipeline.process_mission(output_dir=mission_output_dir)
        processed_mission_outputs.append({
            "mission": mission,
            "mission_path": mission_path,
            "output_dir": mission_output_dir,
        })
    except Exception as e:
        print(f"Error processing {mission_short_name}: {e}")
        erroneous_missions.append((mission, e))

print(f"Processed {len(processed_mission_outputs)} missions.")

if erroneous_missions:
    print("\nErroneous missions:")
    for mission, error in erroneous_missions:
        print(f"- {mission['short_name']}: {error}")

Processing missions:   0%|          | 0/49 [00:00<?, ?it/s]

Skipping ETH-1: Output dir is nonempty
Processing ETH-2 -> output/ETH-2
Using hyperparameters:
P_x=
[[1.5 0.  0. ]
 [0.  1.5 0. ]
 [0.  0.  1. ]]
P_m=
[[0.2 0.  0. ]
 [0.  0.2 0. ]
 [0.  0.  1.5]]
P_w=
[[1.2 0.  0. ]
 [0.  1.2 0. ]
 [0.  0.  1. ]]
N=10
STEP_SIZE=9
dt=0.1
Loading data from /datahome/lmertens/grand_tour_dataset/ros1/ETH-2 ...
Median dt: 0.050s, regular diffs: 5715/5738
Error processing ETH-2: Topic '/boxi/inertial_explorer/tc/odometry' not found in bag /datahome/lmertens/grand_tour_dataset/ros1/ETH-2/2024-10-01-11-47-44_cpt7_ie_tc.bag. Available topics: []
Skipping ETH-3: Output dir is nonempty
Skipping SPX-1: Output dir is nonempty
Skipping SPX-2: Output dir is nonempty
Skipping SPX-3: Output dir is nonempty
Skipping ICE-1: Output dir is nonempty
Skipping SNOW-1: Output dir is nonempty
Skipping SNOW-2: Output dir is nonempty
Skipping SNOW-3: Output dir is nonempty
Skipping EIG-1: Output dir is nonempty
Skipping EIG-2: Output dir is nonempty
Skipping GRI-1: Output dir is